# RE-RL: Генерация данных для Theorem Proving (LeanNavigator)

**Полностью воспроизводимый pipeline** — от Mathlib4 до training pairs.

### Что делает этот notebook

```
Mathlib4 (GitHub)
  │
  ├── [Шаг 1] fast_trace.py — clone, build, ExtractData → .ast.json (кэшируется)
  │
  ├── [Шаг 2] Извлечение шаблонов тактик из 722K тактик (кэшируется)
  │
  ├── [Шаг 3] FAISS RAG index для retrieval тактик (кэшируется)
  │
  ├── [Шаг 4] Загрузка теорем + BFS exploration через Pantograph
  │
  ├── [Шаг 5] Анализ результатов
  │
  └── [Шаг 6] Сохранение датасета (JSONL / SFT / Chat)
```

**Все шаги кэшируются** — при повторном запуске пропускается то, что уже выполнено.

### Зависимости (без lean-dojo)
- `pantograph` — интерактивное доказательство через Lean 4
- `sentence-transformers` — embeddings для RAG
- `faiss-cpu` — vector similarity search
- `elan` — менеджер версий Lean 4
- `re_rl` — наш пакет

---
## 0. Конфигурация

In [ ]:
# ══════════════════════════════════════════════════════════════
# КОНФИГУРАЦИЯ — измените под свои нужды
# ══════════════════════════════════════════════════════════════

# Версия Mathlib4 (полный список: https://github.com/leanprover-community/mathlib4/tags)
MATHLIB_VERSION = "v4.26.0"

# ── Параметры генерации ──
MAX_THEOREMS = 50           # Сколько теорем исследовать BFS (0 = все)
MAX_STEPS_PER_THEOREM = 5000  # Макс шагов BFS на теорему
MAX_TIME_PER_THEOREM = 60     # Секунд на теорему
MIN_TEMPLATE_FREQ = 3         # Минимальная частота шаблона для RAG

# ── Пути (автоматические) ──
import os
from pathlib import Path

CACHE_BASE = Path.home() / ".cache" / "re_rl"
REPO_DIR = CACHE_BASE / f"mathlib4-{MATHLIB_VERSION}" / "mathlib4"
NAV_DATA = CACHE_BASE / f"mathlib4-{MATHLIB_VERSION}" / "navigator_data"
OUTPUT_DIR = Path("./datasets/formal_math_data")
OUTPUT_FORMAT = "jsonl"  # jsonl | json | sft | chat

# elan в PATH
os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ.get("PATH", "")

print(f"Mathlib4:       {MATHLIB_VERSION}")
print(f"Кэш:           {CACHE_BASE / f'mathlib4-{MATHLIB_VERSION}'}")
print(f"Теорем для BFS: {MAX_THEOREMS if MAX_THEOREMS > 0 else 'все'}")
print(f"Формат вывода:  {OUTPUT_FORMAT}")

## 0.1 Проверка зависимостей

In [ ]:
def check_dependencies():
    """Проверяет все зависимости."""
    print("ПРОВЕРКА ЗАВИСИМОСТЕЙ")
    print("=" * 50)
    ok = True

    # elan
    elan = Path.home() / ".elan" / "bin" / "elan"
    if elan.exists():
        print(f"  elan:                  OK")
    else:
        print(f"  elan:                  НЕТ  →  curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh")
        ok = False

    # pantograph
    try:
        import pantograph
        print(f"  pantograph:            OK")
    except ImportError:
        print(f"  pantograph:            НЕТ  →  pip install pantograph")
        ok = False

    # faiss
    try:
        import faiss
        print(f"  faiss-cpu:             OK")
    except ImportError:
        print(f"  faiss-cpu:             НЕТ  →  pip install faiss-cpu")
        ok = False

    # sentence-transformers
    try:
        from sentence_transformers import SentenceTransformer
        print(f"  sentence-transformers: OK")
    except ImportError:
        print(f"  sentence-transformers: НЕТ  →  pip install sentence-transformers")
        ok = False

    # re_rl
    try:
        import re_rl
        print(f"  re_rl:                 OK")
    except ImportError:
        print(f"  re_rl:                 НЕТ  →  pip install -e .")
        ok = False

    print()
    if ok:
        print("Все зависимости установлены!")
    else:
        print("Установите недостающие и перезапустите ячейку.")
    return ok

DEPS_OK = check_dependencies()

In [ ]:
# Раскомментируйте если нужно установить
# !pip install pantograph faiss-cpu sentence-transformers
# !pip install -e ..  # re_rl

## 0.2 Импорты

In [ ]:
import sys
import json
import time
import random
import subprocess
from collections import defaultdict
from datetime import datetime
from typing import List, Dict, Any

# Добавляем корень проекта в PATH если re_rl не установлен через pip
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from re_rl.tasks.formal.lean_navigator import (
    TacticTemplateExtractor,
    TacticRAG,
    PantographDojo,
    LeanNavigatorExplorer,
    TrainingPair,
    load_theorems_from_ast_dir,
    run_lean_navigator,
)

print("Импорты загружены")

---
## Шаг 1: Трейсинг Mathlib4

Скрипт `fast_trace.py` выполняет:
1. `git clone --depth 1` Mathlib4
2. `lake exe cache get` — скачивание предсобранных .olean
3. `lake build` — сборка
4. `ExtractData.lean` — извлечение тактик и посылок

**Все этапы кэшируются** в `~/.cache/re_rl/mathlib4-<version>/`.
При повторном запуске — мгновенный пропуск.

In [ ]:
# Проверяем: трейсинг уже выполнен?
build_ir = REPO_DIR / ".lake" / "build" / "ir"
tracing_done = (REPO_DIR.parent / ".step_5_done").exists()

if tracing_done:
    n_ast = sum(1 for _ in build_ir.rglob("*.ast.json")) if build_ir.exists() else 0
    print(f"Трейсинг уже выполнен! (ast.json: {n_ast})")
    print(f"Данные: {REPO_DIR}")
    print("Пропускаем шаг 1.")
else:
    print(f"Трейсинг НЕ выполнен для {MATHLIB_VERSION}.")
    print(f"Запустите в терминале:")
    print(f"  python scripts/fast_trace.py --version {MATHLIB_VERSION}")
    print()
    print("Или запустите следующую ячейку.")

In [ ]:
# Запуск трейсинга (если ещё не выполнен).
# Занимает ~60 мин для v4.26.0 на 24-ядерной машине.
# При повторном запуске — все этапы пропускаются.

if not tracing_done:
    script = PROJECT_ROOT / "scripts" / "fast_trace.py"
    !python {script} --version {MATHLIB_VERSION}
    
    # Обновляем флаг
    tracing_done = (REPO_DIR.parent / ".step_5_done").exists()
    if tracing_done:
        print("\nТрейсинг завершён!")
    else:
        print("\nТрейсинг не завершился. Проверьте ошибки выше.")
else:
    print("Трейсинг уже выполнен — пропускаем.")

---
## Шаг 2: Извлечение шаблонов тактик

Из 722K тактик строим шаблоны:
- `simp`, `rfl`, `ring` — без параметров
- `rw [{hypothesis}]`, `cases {variable}` — с подстановкой переменных из состояния
- `intro {nvar0}` — с генерацией новых имён

In [ ]:
templates_path = NAV_DATA / "tactic_templates.json"
NAV_DATA.mkdir(parents=True, exist_ok=True)

extractor = TacticTemplateExtractor()

if templates_path.exists():
    extractor.load(str(templates_path))
    print("Загружены из кэша.")
else:
    print("Извлекаем шаблоны из ast.json...")
    t0 = time.time()
    extractor.extract_from_ast_dir(str(REPO_DIR))
    print(f"Время: {time.time()-t0:.1f}с")
    extractor.save(str(templates_path))

print(f"\nТоп-15 шаблонов тактик:")
for tmpl, freq in extractor.get_top_templates(15):
    print(f"  [{freq:6d}] {tmpl[:65]}")

---
## Шаг 3: FAISS RAG index

Строим FAISS index для retrieval шаблонов тактик по текущему proof state.
Используем `all-MiniLM-L6-v2` (sentence-transformers) для embeddings.

In [ ]:
rag_path = NAV_DATA / "rag_index"

rag = TacticRAG(model_name="all-MiniLM-L6-v2")

if (rag_path / "faiss.index").exists():
    rag.load(str(rag_path))
    print("Загружен из кэша.")
else:
    print("Строим FAISS index...")
    t0 = time.time()
    rag.build_index(extractor.templates, min_freq=MIN_TEMPLATE_FREQ)
    print(f"Время: {time.time()-t0:.1f}с")
    rag.save(str(rag_path))

---
## Шаг 4: BFS exploration через Pantograph

Для каждой теоремы:
1. RAG → top-200 шаблонов тактик для текущего состояния
2. Шаблоны → конкретные тактики (подстановка переменных)
3. Каждая тактика → `run_tac` через Pantograph
4. Новые состояния → приоритетная очередь (BFS)
5. `ProofFinished` → записываем путь, генерируем training pairs

In [ ]:
# Загрузка теорем из traced данных
print("Загрузка теорем из ast данных...")
t0 = time.time()
all_theorems = load_theorems_from_ast_dir(str(REPO_DIR))
print(f"Время: {time.time()-t0:.1f}с")

# Фильтруем: нужны теоремы с тактическими доказательствами
all_theorems = [t for t in all_theorems if len(t.tactics) >= 2]
print(f"\nТеорем с тактическими доказательствами (>=2 шага): {len(all_theorems)}")

# Выборка для BFS
if MAX_THEOREMS > 0 and len(all_theorems) > MAX_THEOREMS:
    theorems = random.sample(all_theorems, MAX_THEOREMS)
else:
    theorems = all_theorems

print(f"Выбрано для BFS: {len(theorems)}")
print(f"\nПримеры:")
for t in theorems[:5]:
    print(f"  {t.name[:60]}  |  tacs={len(t.tactics)}  |  goal={t.goal_expr[:50]}...")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  BFS EXPLORATION
# ══════════════════════════════════════════════════════════════

all_pairs = []
theorem_results = []
total_start = time.time()

print(f"Запускаем BFS на {len(theorems)} теоремах...")
print(f"  max_steps={MAX_STEPS_PER_THEOREM}, max_time={MAX_TIME_PER_THEOREM}s")
print()

with PantographDojo(project_path=str(REPO_DIR), imports=["Mathlib"]) as dojo:
    explorer = LeanNavigatorExplorer(
        dojo=dojo, rag=rag,
        max_steps=MAX_STEPS_PER_THEOREM,
        max_time=MAX_TIME_PER_THEOREM,
        verbose=False,
    )

    # Разрешаем типы через env_inspect
    resolved = 0
    for thm in theorems:
        full_type = dojo.env_inspect(thm.name)
        if full_type:
            thm.goal_expr = full_type
            resolved += 1
    print(f"Разрешено типов через env_inspect: {resolved}/{len(theorems)}")
    print()

    for i, thm in enumerate(theorems):
        t0 = time.time()
        try:
            result = explorer.explore(
                goal_expr=thm.goal_expr,
                theorem_name=thm.name,
                theorem_code=thm.goal_state,
                exit_on_finish=False,
            )

            all_pairs.extend(result.pairs)
            elapsed = time.time() - t0
            status = "PROVEN" if result.theorem_proven else "------"
            theorem_results.append({
                "theorem": thm.name,
                "proven": result.theorem_proven,
                "states": result.n_states,
                "steps": result.n_steps,
                "pairs": len(result.pairs),
                "time": elapsed,
            })

            if (i + 1) % 10 == 0 or result.theorem_proven:
                proven_so_far = sum(1 for r in theorem_results if r["proven"])
                total_pairs_so_far = sum(r["pairs"] for r in theorem_results)
                print(f"  [{i+1:4d}/{len(theorems)}] {status} | "
                      f"proven={proven_so_far} pairs={total_pairs_so_far} | "
                      f"{thm.name[:45]} ({elapsed:.1f}s)")

        except Exception as e:
            theorem_results.append({
                "theorem": thm.name, "proven": False,
                "error": str(e)[:80], "pairs": 0,
                "states": 0, "steps": 0, "time": 0,
            })

total_time = time.time() - total_start
proofs_found = sum(1 for r in theorem_results if r.get("proven"))

print(f"\n{'='*60}")
print(f"  ИТОГО")
print(f"{'='*60}")
print(f"  Теорем исследовано:     {len(theorem_results)}")
print(f"  Доказательств найдено:  {proofs_found}")
print(f"  Training pairs:         {len(all_pairs)}")
print(f"  Уникальных тактик:      {len(set(p.tactic for p in all_pairs)) if all_pairs else 0}")
print(f"  Время:                  {total_time:.1f}с ({total_time/60:.1f} мин)")

---
## Шаг 5: Анализ результатов

In [ ]:
print("АНАЛИЗ TRAINING PAIRS")
print("=" * 50)
print(f"Всего pairs: {len(all_pairs)}")

if all_pairs:
    # Уникальные тактики
    unique_tactics = set(p.tactic for p in all_pairs)
    print(f"Уникальных тактик: {len(unique_tactics)}")

    # Распределение по distance_to_proof
    dist_counts = defaultdict(int)
    for p in all_pairs:
        dist_counts[p.distance_to_proof] += 1
    print(f"\nРаспределение по distance_to_proof:")
    for d in sorted(dist_counts.keys())[:10]:
        print(f"  distance={d}: {dist_counts[d]} pairs")

    # Топ тактики
    tactic_counts = defaultdict(int)
    for p in all_pairs:
        base_tac = p.tactic.split(' ')[0] if ' ' in p.tactic else p.tactic
        tactic_counts[base_tac] += 1

    print(f"\nТоп-15 тактик:")
    for tactic, count in sorted(tactic_counts.items(), key=lambda x: -x[1])[:15]:
        pct = 100 * count / len(all_pairs)
        print(f"  {tactic:25s} {count:5d} ({pct:.1f}%)")

    # Доказанные теоремы
    proven = [r for r in theorem_results if r.get("proven")]
    if proven:
        print(f"\nДоказанные теоремы ({len(proven)}):")
        for r in proven[:20]:
            print(f"  {r['theorem'][:55]:55s}  steps={r['steps']:5d}  pairs={r['pairs']}")
else:
    print("Нет training pairs. Проверьте шаг 4.")

In [ ]:
# Примеры training pairs
if all_pairs:
    print("ПРИМЕРЫ TRAINING PAIRS")
    print("=" * 50)

    for i, pair in enumerate(all_pairs[:8]):
        print(f"\n--- Pair {i+1} (distance={pair.distance_to_proof}) ---")
        print(f"Theorem: {pair.theorem_name}")
        state_lines = pair.state.split('\n')
        for line in state_lines[:4]:
            print(f"  {line}")
        if len(state_lines) > 4:
            print(f"  ... (+{len(state_lines)-4} строк)")
        print(f"Tactic:  {pair.tactic}")
        print(f"Next:    {pair.next_state[:80]}..." if len(pair.next_state) > 80 else f"Next:    {pair.next_state}")

---
## Шаг 6: Сохранение датасета

In [ ]:
def save_dataset(pairs, output_dir, fmt, metadata):
    """Сохраняет датасет."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    def pair_to_dict(p):
        return {
            "state": p.state, "tactic": p.tactic,
            "next_state": p.next_state,
            "distance_to_proof": p.distance_to_proof,
            "theorem_name": p.theorem_name,
        }

    if fmt == "jsonl":
        f = output_dir / f"lean_data_{ts}.jsonl"
        with open(f, "w") as fh:
            for p in pairs:
                fh.write(json.dumps(pair_to_dict(p), ensure_ascii=False) + "\n")
    elif fmt == "json":
        f = output_dir / f"lean_data_{ts}.json"
        with open(f, "w") as fh:
            json.dump([pair_to_dict(p) for p in pairs], fh, indent=2, ensure_ascii=False)
    elif fmt == "sft":
        f = output_dir / f"lean_sft_{ts}.json"
        sft = [{
            "instruction": "You are a Lean 4 theorem prover. Given the current proof state, suggest the next tactic.",
            "input": f"Current proof state:\n{p.state}",
            "output": p.tactic,
        } for p in pairs]
        with open(f, "w") as fh:
            json.dump(sft, fh, indent=2, ensure_ascii=False)
    elif fmt == "chat":
        f = output_dir / f"lean_chat_{ts}.json"
        chat = [{
            "messages": [
                {"role": "system", "content": "You are an expert Lean 4 theorem prover."},
                {"role": "user", "content": f"Prove this goal:\n```\n{p.state}\n```"},
                {"role": "assistant", "content": p.tactic},
            ]
        } for p in pairs]
        with open(f, "w") as fh:
            json.dump(chat, fh, indent=2, ensure_ascii=False)
    else:
        raise ValueError(f"Unknown format: {fmt}")

    # Метаданные
    mf = output_dir / f"metadata_{ts}.json"
    with open(mf, "w") as fh:
        json.dump(metadata, fh, indent=2, ensure_ascii=False, default=str)

    print(f"Датасет:   {f}  ({f.stat().st_size:,} bytes)")
    print(f"Метаданные: {mf}")
    return f


if all_pairs:
    metadata = {
        "mathlib_version": MATHLIB_VERSION,
        "repo_dir": str(REPO_DIR),
        "generation_date": datetime.now().isoformat(),
        "config": {
            "max_theorems": MAX_THEOREMS,
            "max_steps_per_theorem": MAX_STEPS_PER_THEOREM,
            "max_time_per_theorem": MAX_TIME_PER_THEOREM,
            "min_template_freq": MIN_TEMPLATE_FREQ,
        },
        "summary": {
            "total_theorems": len(theorem_results),
            "proofs_found": proofs_found,
            "total_pairs": len(all_pairs),
            "unique_tactics": len(set(p.tactic for p in all_pairs)),
            "total_time": total_time,
        },
    }

    output_file = save_dataset(all_pairs, OUTPUT_DIR, OUTPUT_FORMAT, metadata)
else:
    print("Нет данных для сохранения.")

---
## Альтернатива: Полный pipeline одной командой

Если не нужен пошаговый контроль — можно запустить всё разом через `run_lean_navigator()`:

In [ ]:
# # Полный pipeline одной командой (раскомментируйте):
# all_pairs, summary = run_lean_navigator(
#     repo_dir=str(REPO_DIR),
#     max_theorems=50,
#     max_steps=5000,
#     max_time=60,
#     min_template_freq=3,
#     output_dir=str(OUTPUT_DIR / "full_run"),
#     templates_path=str(templates_path),    # используем кэш
#     rag_path=str(rag_path),                # используем кэш
#     imports=["Mathlib"],
#     verbose=False,
# )

---
## Быстрый тест: Pantograph + конкретные теоремы

Можно быстро проверить BFS на конкретных теоремах:

In [ ]:
# Быстрый тест на конкретных теоремах
test_goals = [
    ("Nat.add_comm", "\u2200 (n m : \u2115), n + m = m + n"),
    ("Nat.zero_le",  "\u2200 (n : \u2115), 0 \u2264 n"),
    ("Int.add_comm", "\u2200 (a b : \u2124), a + b = b + a"),
    ("Nat.succ_pos", "\u2200 (n : \u2115), 0 < n + 1"),
]

with PantographDojo(project_path=str(REPO_DIR), imports=["Mathlib"]) as dojo:
    quick_explorer = LeanNavigatorExplorer(
        dojo=dojo, rag=rag,
        max_steps=3000, max_time=15, verbose=True,
    )
    for name, goal in test_goals:
        print(f"\n{'='*50}")
        print(f"[BFS] {name}: {goal}")
        result = quick_explorer.explore(
            goal_expr=goal, theorem_name=name,
            theorem_code=goal, exit_on_finish=True,
        )
        s = "PROVEN" if result.theorem_proven else "not proven"
        print(f"  {s} | states={result.n_states} steps={result.n_steps} "
              f"pairs={len(result.pairs)} time={result.elapsed:.1f}s")

---
## Справка

### Структура кэша
```
~/.cache/re_rl/mathlib4-v4.26.0/
  ├── .step_1_done ... .step_7_done   # маркеры этапов трейсинга
  ├── mathlib4/                         # исходники + build
  │   ├── Mathlib/                      # .lean файлы
  │   └── .lake/build/ir/              # .ast.json, .dep_paths
  └── navigator_data/                   # кэш шаблонов и RAG
      ├── tactic_templates.json         # 401K шаблонов
      └── rag_index/                    # FAISS index
```

### Команды
```bash
# Трейсинг (первый раз ~60 мин, потом мгновенно)
python scripts/fast_trace.py --version v4.26.0

# Перезапуск с конкретного этапа
python scripts/fast_trace.py --version v4.26.0 --force-step 5

# Другая версия Mathlib
python scripts/fast_trace.py --version v4.28.0
```

### Масштабирование
| Параметр | Быстрый тест | Средний | Полный |
|----------|-------------|---------|--------|
| MAX_THEOREMS | 10 | 100 | 0 (все) |
| MAX_STEPS | 3000 | 10000 | 200000 |
| MAX_TIME | 15с | 60с | 1200с |
| Ожидаемое время | 2 мин | 30 мин | дни |
| Ожидаемых pairs | 10-50 | 500-2000 | миллионы |

In [ ]:
print("=" * 50)
print("ГОТОВО!")
print("=" * 50)
if all_pairs:
    print(f"\nСгенерировано {len(all_pairs)} training pairs")
    print(f"из {len(theorem_results)} теорем ({proofs_found} доказано).")
    print(f"Датасет: {OUTPUT_DIR}")
else:
    print("\nДанные не сгенерированы — проверьте шаги выше.")
print(f"\nДля масштабирования увеличьте MAX_THEOREMS и MAX_STEPS.")